# Normalising Timeseries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import os
import numpy as np
import pandas as pd
import xarray as xr
import itertools
import geopandas as gpd
import re

import pickle

# from matplotlib_venn import venn3
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.preprocessing import StandardScaler
import glob
import gc


In [ ]:
# ============================================================================
# PART 1: Load and normalize hydromet data
# ============================================================================
folder_path = '/perm/mokr/CamelsGB_V2/Catchment_Timeseries/hydro-meteorological/daily'
pattern = "camels_gb_v2_hydromet_daily_timeseries_*.csv"

csv_files = glob.glob(str(Path(folder_path) / pattern))
print(f"Found {len(csv_files)} CSV files")

# First pass: determine columns and fit scaler
print("First pass: fitting scaler...")
scaler_hydromet = StandardScaler()
first_file = True

for i, file in enumerate(csv_files):
    if i % 50 == 0:
        print(f"Processing file {i+1}/{len(csv_files)} for scaler fitting")
    
    df = pd.read_csv(file)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if first_file:
        numeric_cols_hydromet = numeric_cols
        first_file = False
    
    scaler_hydromet.partial_fit(df[numeric_cols_hydromet])
    # del df
    gc.collect()

print(f"Numeric columns to normalize: {numeric_cols_hydromet}")

In [22]:
import joblib
joblib.dump(scaler_hydromet, '/home/mokr/Reservoir_Modelling/scaler_hydromet.pkl')

['/home/mokr/Reservoir_Modelling/scaler_hydromet.pkl']

In [ ]:
# Second pass: create normalized dictionary
print("\nSecond pass: creating normalized dictionary...")
hydromet_normalized_dict = {}

for i, file in enumerate(csv_files):
    if i % 50 == 0:
        print(f"Loading and normalizing file {i+1}/{len(csv_files)}")
    
    df = pd.read_csv(file)
    
    # Extract gauge_id from filename
    filename = Path(file).name
    parts = filename.split('_')
    gauge_id = parts[6]
    
    # Normalize numeric columns
    df[numeric_cols_hydromet] = scaler_hydromet.transform(df[numeric_cols_hydromet])
    
    # Store in dictionary with gauge_id as key
    hydromet_normalized_dict[gauge_id] = df
    
    del df
    gc.collect()

print(f"\nCreated hydromet_normalized_dict with {len(hydromet_normalized_dict)} gauges")

In [9]:
df

,date,precipitation_cehgear,precipitation_haduk,pet_chess,peti_chess,pet_hydrope,peti_hydrope,temperature_chess,temperature_haduk,discharge_spec,discharge_vol
0,1970-10-01,0.00,0.01,1.42,1.42,1.69,1.70,12.81,13.50,NaN,NaN
1,1970-10-02,0.02,0.03,2.65,2.66,1.10,1.12,12.60,11.46,NaN,NaN
2,1970-10-03,1.99,2.03,2.31,2.58,1.28,1.58,11.53,12.03,NaN,NaN
3,1970-10-04,0.01,0.07,0.27,0.28,2.04,2.06,12.61,14.64,NaN,NaN
4,1970-10-05,2.53,2.78,0.74,0.98,0.69,1.00,13.12,9.81,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
18988,2022-09-26,NaN,0.18,NaN,NaN,1.01,1.09,NaN,12.41,0.30,0.05
18989,2022-09-27,NaN,1.39,NaN,NaN,0.58,0.97,NaN,10.96,0.27,0.05
18990,2022-09-28,NaN,9.38,NaN,NaN,0.52,0.89,NaN,9.76,0.31,0.06
18991,2022-09-29,NaN,0.04,NaN,NaN,0.51,0.53,NaN,9.73,0.33,0.06


In [16]:
scaler_hydromet

,copy,True
,with_mean,True
,with_std,True


In [9]:
# ============================================================================
# PART 2: Load and normalize temporal data from pickle
# ============================================================================
pickle_file_path = '/home/mokr/Camels_GB/Output_CSVs/GB_Temporal_Dictionary.pkl'
with open(pickle_file_path, 'rb') as pickle_file:
    Gauge_Data = pickle.load(pickle_file)

print(f"\nProcessing {len(Gauge_Data)} gauges from pickle file...")

# Fit scaler on temporal data
scaler_temporal = StandardScaler()
for gauge_id, df in Gauge_Data.items():
    numeric_cols_temporal = df.select_dtypes(include=[np.number]).columns.tolist()
    scaler_temporal.partial_fit(df[numeric_cols_temporal])
    break  # Get columns from first gauge

# Normalize all temporal dataframes
for gauge_id, df in Gauge_Data.items():
    scaler_temporal.partial_fit(df[numeric_cols_temporal])

print(f"Temporal numeric columns: {numeric_cols_temporal}")

# Create normalized temporal dictionary
temporal_normalized_dict = {}
for gauge_id, df in Gauge_Data.items():
    df_normalized = df.copy()
    df_normalized[numeric_cols_temporal] = scaler_temporal.transform(df[numeric_cols_temporal])
    temporal_normalized_dict[gauge_id] = df_normalized

print(f"Created temporal_normalized_dict with {len(temporal_normalized_dict)} gauges")

# Clean up original Gauge_Data to free memory
del Gauge_Data
gc.collect()

# ============================================================================
# PART 3: Merge the two dictionaries
# ============================================================================
print("\nMerging hydromet and temporal data...")
merged_dict = {}

# Find common gauge_ids
hydromet_ids = set(hydromet_normalized_dict.keys())
temporal_ids = set(temporal_normalized_dict.keys())
common_ids = hydromet_ids & temporal_ids

print(f"Hydromet gauges: {len(hydromet_ids)}")
print(f"Temporal gauges: {len(temporal_ids)}")
print(f"Common gauges: {len(common_ids)}")

# Merge dataframes for each gauge_id
for gauge_id in common_ids:
    # Assuming there's a common column to merge on (e.g., 'date' or index)
    # Adjust merge parameters based on your data structure
    merged_df = pd.merge(
        hydromet_normalized_dict[gauge_id],
        temporal_normalized_dict[gauge_id],
        left_index=True,
        right_index=True,
        how='inner'  # Change to 'outer', 'left', or 'right' as needed
    )
    merged_dict[gauge_id] = merged_df

print(f"\nCreated merged_dict with {len(merged_dict)} gauges")

# Clean up intermediate dictionaries
del hydromet_normalized_dict, temporal_normalized_dict
gc.collect()

# ============================================================================
# Verification
# ============================================================================
if merged_dict:
    first_gauge = list(merged_dict.keys())[0]
    print(f"\nVerification for gauge {first_gauge}:")
    print(f"  Merged dataframe shape: {merged_dict[first_gauge].shape}")
    print(f"  Columns: {list(merged_dict[first_gauge].columns)}")
    print(f"\n  First few rows:")
    print(merged_dict[first_gauge].head())

print("\nProcessing complete!")
print(f"Final dictionary 'merged_dict' contains {len(merged_dict)} gauge dataframes")


Processing 671 gauges from pickle file...
Temporal numeric columns: ['precipitation', 'pet', 'temperature', 'discharge_spec', 'discharge_vol', 'peti', 'humidity', 'shortwave_rad', 'longwave_rad', 'windspeed']
Created temporal_normalized_dict with 671 gauges

Merging hydromet and temporal data...
Hydromet gauges: 671
Temporal gauges: 671
Common gauges: 671

Created merged_dict with 671 gauges

Verification for gauge 34010:
  Merged dataframe shape: (16436, 22)
  Columns: ['date_x', 'precipitation_cehgear', 'precipitation_haduk', 'pet_chess', 'peti_chess', 'pet_hydrope', 'peti_hydrope', 'temperature_chess', 'temperature_haduk', 'discharge_spec_x', 'discharge_vol_x', 'date_y', 'precipitation', 'pet', 'temperature', 'discharge_spec_y', 'discharge_vol_y', 'peti', 'humidity', 'shortwave_rad', 'longwave_rad', 'windspeed']

  First few rows:
       date_x  precipitation_cehgear  precipitation_haduk  pet_chess  \
0  1970-10-01              -0.511611            -0.513338   0.592269   
1  1970-1

In [12]:
def check_and_consolidate_duplicate_columns(data_dict, col1, col2, keep='first', rename_to=None):
    """
    Check if two columns are identical across all dataframes in a dictionary.
    Optionally consolidate them if identical.
    
    Parameters:
    -----------
    data_dict : dict
        Dictionary with gauge_id as keys and dataframes as values
    col1 : str
        First column name (e.g., 'date_x')
    col2 : str
        Second column name (e.g., 'date_y')
    keep : str, default='first'
        Which column to keep if identical: 'first', 'second', or None (just check)
    rename_to : str, optional
        New name for the kept column (e.g., 'date')
    
    Returns:
    --------
    bool : True if all columns are identical, False otherwise
    """
    print(f"\nChecking {col1} vs {col2}...")
    
    all_identical = True
    mismatched_gauges = []
    
    for gauge_id, df in data_dict.items():
        if col1 in df.columns and col2 in df.columns:
            # Check if all values are identical (accounting for NaN)
            are_identical = ((df[col1] == df[col2]) | 
                           (df[col1].isna() & df[col2].isna())).all()
            
            if not are_identical:
                all_identical = False
                mismatched_gauges.append(gauge_id)
                
                # Show details for first few mismatches
                if len(mismatched_gauges) <= 3:
                    mismatches = df[~((df[col1] == df[col2]) | 
                                     (df[col1].isna() & df[col2].isna()))][[col1, col2]]
                    print(f"\n  Gauge {gauge_id}: {len(mismatches)} mismatched values")
                    print(f"  First few mismatches:")
                    print(mismatches.head())
                    
                    # Show stats for numeric columns
                    if df[col1].dtype in [np.float64, np.int64, np.float32, np.int32]:
                        print(f"  Stats - {col1}: mean={df[col1].mean():.4f}, std={df[col1].std():.4f}")
                        print(f"  Stats - {col2}: mean={df[col2].mean():.4f}, std={df[col2].std():.4f}")
                        print(f"  Correlation: {df[[col1, col2]].corr().iloc[0,1]:.6f}")
        elif col1 not in df.columns:
            print(f"  Warning: {col1} not found in gauge {gauge_id}")
        elif col2 not in df.columns:
            print(f"  Warning: {col2} not found in gauge {gauge_id}")
    
    if all_identical:
        print(f"✓ All {col1} and {col2} columns are identical across all gauges!")
        
        # Consolidate if requested
        if keep is not None:
            print(f"\nConsolidating columns (keeping '{keep}')...")
            
            col_to_keep = col1 if keep == 'first' else col2
            col_to_drop = col2 if keep == 'first' else col1
            
            for gauge_id in data_dict.keys():
                if col_to_drop in data_dict[gauge_id].columns:
                    data_dict[gauge_id] = data_dict[gauge_id].drop(columns=[col_to_drop])
                    
                    # Rename if requested
                    if rename_to is not None and col_to_keep in data_dict[gauge_id].columns:
                        data_dict[gauge_id] = data_dict[gauge_id].rename(columns={col_to_keep: rename_to})
            
            final_name = rename_to if rename_to else col_to_keep
            print(f"✓ Consolidated to single '{final_name}' column")
    else:
        print(f"\n✗ Found {len(mismatched_gauges)} gauges with mismatched values:")
        print(f"  {mismatched_gauges[:10]}{'...' if len(mismatched_gauges) > 10 else ''}")
    
    # Quick sample check on first gauge
    if data_dict:
        first_gauge = list(data_dict.keys())[0]
        sample_df = data_dict[first_gauge]
        if col1 in sample_df.columns and col2 in sample_df.columns:
            print(f"\nSample from gauge {first_gauge}:")
            print(sample_df[[col1, col2]].head(10))
    
    return all_identical


# Usage examples:
# # Just check without consolidating
# check_and_consolidate_duplicate_columns(merged_dict, 'date_x', 'date_y', keep=None)

# # Check and consolidate, keeping first column and renaming
# check_and_consolidate_duplicate_columns(merged_dict, 'date_x', 'date_y', keep='first', rename_to='date')

# Check and consolidate discharge columns, keeping discharge_vol
check_and_consolidate_duplicate_columns(merged_dict, 'discharge_vol_x', 'discharge_vol_y', keep='first')

# Or keep second column
check_and_consolidate_duplicate_columns(merged_dict, 'discharge_spec_x', 'discharge_spec_y', keep='second')


Checking discharge_vol_x vs discharge_vol_y...

  Gauge 54007: 16381 mismatched values
  First few mismatches:
   discharge_vol_x  discharge_vol_y
0        -0.286924        -0.288419
1        -0.288185        -0.289687
2        -0.290708        -0.292222
3        -0.294071        -0.295602
4        -0.290708        -0.292222
  Stats - discharge_vol_x: mean=-0.2156, std=0.1896
  Stats - discharge_vol_y: mean=-0.2178, std=0.1855
  Correlation: 0.997462

  Gauge 11004: 11115 mismatched values
  First few mismatches:
      discharge_vol_x  discharge_vol_y
4909        -0.068719        -0.069138
4910        -0.037186        -0.037450
4911         0.018311         0.018321
4912        -0.061151        -0.061533
4913        -0.061151        -0.061533
  Stats - discharge_vol_x: mean=-0.1975, std=0.1257
  Stats - discharge_vol_y: mean=-0.1986, std=0.1264
  Correlation: 1.000000

  Gauge 96001: 14518 mismatched values
  First few mismatches:
      discharge_vol_x  discharge_vol_y
1918        -0.

False

In [14]:

# output_dir = Path('/home/mokr/Reservoir_Modelling/')
# output_file = output_dir / 'Normalised_Temporal_Data.pkl'

# # Create directory if it doesn't exist
# output_dir.mkdir(parents=True, exist_ok=True)

# # Save the dictionary
# print(f"\nSaving merged_dict to {output_file}...")
# with open(output_file, 'wb') as f:
#     pickle.dump(merged_dict, f, protocol=pickle.HIGHEST_PROTOCOL)


Saving merged_dict to /home/mokr/Reservoir_Modelling/Normalised_Temporal_Data.pkl...
